### Supplement 3: Tools, also called function calling (`3-tools.py`)

The model answers a weather question by asking **your code** to run `get_weather`.

**The key point:** The model can't run code; it can only write text. "Calling a tool" means the model writes a request (a function name plus JSON arguments), your code runs the function, and a second API call gives the model the result so it can write the answer.

```
Call 1     create(messages, tools)
           └─> a request, not an answer: run get_weather(lat, lon)
Your code  json.loads(arguments) -> get_weather(**args) -> Open-Meteo
           └─> append the request and the result to messages
Call 2     parse(messages, tools, response_format=WeatherResponse)
           └─> the answer, as a WeatherResponse
```

Deep dive: [Exhaustive_3-tools.ipynb](Exhaustive_3-tools.ipynb). OpenAI docs: [function calling](https://platform.openai.com/docs/guides/function-calling).

#### 1. Setup
`OpenAI()` with no `api_key` reads `OPENAI_API_KEY` from the environment by itself.

In [1]:
import json
import requests
from openai import OpenAI
from pydantic import BaseModel, Field
from dotenv import load_dotenv

load_dotenv()

client = OpenAI()

#### 2. The function (it runs on your machine)
An ordinary Python function: it asks the free Open-Meteo API for the current weather at the given coordinates, in the location's local time (`timezone=auto`) and in m/s (`wind_speed_unit=ms`). It returns the values **together with their units and timezone**, because this result is all the model will know about the weather. The model never sees this code or its docstring.

`3-tools.py` originally returned only `data["current"]`, the bare numbers, and the model guessed the units and the local time (Section 7).

In [2]:
def get_weather(latitude, longitude):
    """This is a publically available API that returns the weather for a given location."""
    response = requests.get(
        f"https://api.open-meteo.com/v1/forecast?latitude={latitude}&longitude={longitude}&current=temperature_2m,wind_speed_10m&hourly=temperature_2m,relative_humidity_2m,wind_speed_10m&timezone=auto&wind_speed_unit=ms"
    )
    data = response.json()
    # send the units and timezone too, so the model doesn't have to guess them
    return {
        "current": data["current"],
        "units": data["current_units"],
        "timezone": data["timezone"],
    }

#### 3. The tool definition (what the model reads)
This dict is everything the model learns about the tool:
- `name`: the label the model writes back when it wants this tool.
- `description`: what the model reads to decide whether the tool fits the question.
- `parameters`: a JSON Schema for the arguments, with their names and types.
- `"strict": True`: the model's arguments are guaranteed to match that schema. `parse()` in call 2 requires it, and raises a `ValueError` for a tool without it.

In [3]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get current temperature for provided coordinates in celsius.",
            "parameters": {
                "type": "object",
                "properties": {
                    "latitude": {"type": "number"},
                    "longitude": {"type": "number"},
                },
                "required": ["latitude", "longitude"],
                "additionalProperties": False,
            },
            "strict": True,
        },
    }
]

#### 4. Call 1: the model asks for the tool
Same `create()` as Supplement 1, plus `tools=tools`. The reply is a request, not an answer:
- `finish_reason` is `'tool_calls'`: the model stopped because it wants a tool run.
- `content` is `None`: there is no answer text.
- `arguments` is a **string** of JSON, not a dict. Nothing in the request gave Oslo's coordinates; the model wrote them from its own knowledge.
- `id` is what pairs this request with the result you send back.

In [4]:
messages = [
    {"role": "system", "content": "You are a helpful weather assistant."},
    {"role": "user", "content": "What's the weather like in Oslo today?"},
]

completion = client.chat.completions.create(
    model="gpt-5-nano",
    messages=messages,
    tools=tools,
)

In [5]:
print(type(completion))
print(completion.model_dump_json(indent=2))

<class 'openai.types.chat.chat_completion.ChatCompletion'>
{
  "id": "chatcmpl-EPvakSxcU1fhahiIqBCXrTDqXQEbI",
  "choices": [
    {
      "finish_reason": "tool_calls",
      "index": 0,
      "logprobs": null,
      "message": {
        "content": null,
        "refusal": null,
        "role": "assistant",
        "annotations": [],
        "audio": null,
        "function_call": null,
        "tool_calls": [
          {
            "id": "call_VDE7xaiUFHEdx3XuHFoN5dyp",
            "function": {
              "arguments": "{\"latitude\":59.9139,\"longitude\":10.7522}",
              "name": "get_weather"
            },
            "type": "function"
          }
        ]
      }
    }
  ],
  "created": 1789848290,
  "model": "gpt-5-nano-2025-08-07",
  "object": "chat.completion",
  "metadata": null,
  "moderation": null,
  "service_tier": "default",
  "system_fingerprint": null,
  "usage": {
    "completion_tokens": 353,
    "prompt_tokens": 150,
    "total_tokens": 503,
    "complet

In [6]:
reply = completion.choices[0].message   # named `reply`, NOT `message`, to keep it distinct from your `messages` list

print("finish_reason:", completion.choices[0].finish_reason)
print("content:      ", reply.content)
for tool_call in reply.tool_calls:
    print("tool call:")
    print("   id:        ", tool_call.id)
    print("   name:      ", tool_call.function.name)
    print("   arguments: ", repr(tool_call.function.arguments), f"({type(tool_call.function.arguments).__name__})")

finish_reason: tool_calls
content:       None
tool call:
   id:         call_VDE7xaiUFHEdx3XuHFoN5dyp
   name:       get_weather
   arguments:  '{"latitude":59.9139,"longitude":10.7522}' (str)


#### 5. Your code runs the tool
- `call_function` maps the name the model wrote to the real function, because you can't call a string.
- `json.loads` turns the arguments string into a dict, and `**args` passes that dict as keyword arguments: `get_weather(latitude=..., longitude=...)`.
- The model's request (the assistant message from call 1) is appended **once, before the loop**. Then each tool call gets its own `"tool"` message with the matching `tool_call_id`. `json.dumps` turns the result into a string, because a message's `content` must be text.
- Why before the loop: when the model asks for two tools in one reply (e.g. Oslo and Paris), appending inside the loop adds the request twice, and the API rejects call 2 with a 400 error. `3-tools.py` originally had it inside the loop.

The API remembers nothing between calls, so these 4 messages are the entire conversation the model sees in call 2: your system message, your user message, the model's request, and the tool result. The model's request is an SDK object, not a dict; the SDK converts it to JSON when it sends the request.

In [7]:
def call_function(name, args):
    if name == "get_weather":
        return get_weather(**args)


messages.append(completion.choices[0].message)  # the model's request: once, not once per tool call

for tool_call in completion.choices[0].message.tool_calls:
    name = tool_call.function.name
    args = json.loads(tool_call.function.arguments)

    result = call_function(name, args)
    messages.append(
        {"role": "tool", "tool_call_id": tool_call.id, "content": json.dumps(result)}
    )

In [8]:
print("name:  ", name, f"({type(name).__name__})")
print("args:  ", args, f"({type(args).__name__})")
print("result:", result, f"({type(result).__name__})")

name:   get_weather (str)
args:   {'latitude': 59.9139, 'longitude': 10.7522} (dict)
result: {'current': {'time': '2026-09-19T22:00', 'interval': 900, 'temperature_2m': 15.2, 'wind_speed_10m': 5.3}, 'units': {'time': 'iso8601', 'interval': 'seconds', 'temperature_2m': '°C', 'wind_speed_10m': 'm/s'}, 'timezone': 'Europe/Oslo'} (dict)


#### 6. Call 2: the model writes the answer
- `WeatherResponse` is the shape of the final answer. Its `Field(description=...)` texts go into the JSON Schema, so the model reads them.
- `messages` (all 4 of them) and `tools` are sent again, because every request has to carry everything.
- This is the same `parse()` call as in Supplement 2, plus `tools=tools`.

In [10]:
class WeatherResponse(BaseModel):
    temperature: float = Field(
        description="The current temperature in celsius for the given location."
    )
    response: str = Field(
        description="A natural language response to the user's question."
    )

In [11]:
completion_2 = client.chat.completions.parse(
    model="gpt-5-nano",
    messages=messages,
    tools=tools,
    response_format=WeatherResponse,
)

In [12]:
print(type(completion_2))
# warnings=False hides a harmless Pydantic warning about the `parsed` field; it is still printed.
print(completion_2.model_dump_json(indent=2, warnings=False))

<class 'openai.types.chat.parsed_chat_completion.ParsedChatCompletion[TypeVar]'>
{
  "id": "chatcmpl-EPvao2KrIQ8qYysthpo6vwvMkhX5W",
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "logprobs": null,
      "message": {
        "content": "{\"temperature\":15.2,\"response\":\"Right now in Oslo (local time 22:00), it's about 15.2°C with a light breeze (wind around 5.3 m/s).\"}",
        "refusal": null,
        "role": "assistant",
        "annotations": [],
        "audio": null,
        "function_call": null,
        "tool_calls": null,
        "parsed": {
          "temperature": 15.2,
          "response": "Right now in Oslo (local time 22:00), it's about 15.2°C with a light breeze (wind around 5.3 m/s)."
        }
      }
    }
  ],
  "created": 1789848294,
  "model": "gpt-5-nano-2025-08-07",
  "object": "chat.completion",
  "metadata": null,
  "moderation": null,
  "service_tier": "default",
  "system_fingerprint": null,
  "usage": {
    "completion_token

#### 7. The final answer
`parsed` is a `WeatherResponse`, the shape you asked for. The units and the local time in the sentence come from the tool result, not from a guess.

In [13]:
final_response = completion_2.choices[0].message.parsed

print("type:       ", type(final_response).__name__)
print("temperature:", final_response.temperature)
print("response:   ", final_response.response)

type:        WeatherResponse
temperature: 15.2
response:    Right now in Oslo (local time 22:00), it's about 15.2°C with a light breeze (wind around 5.3 m/s).


#### In short
The model can't run code, and the API remembers nothing between calls. So one tool use is two calls with your code in between:

1. **Call 1** sends the question and the tool definition. The model replies with a *request* — the name `get_weather`, its arguments as a JSON **string**, and an `id` — not an answer.
2. **Your code** turns that string into a dict, runs the function, then appends the model's request once and one `"tool"` message per call, each tagged with the matching `id`.
3. **Call 2** sends all four messages again, plus `response_format=WeatherResponse`. The model now has the data, so it writes the answer.

**Why `get_weather` returns `units` and `timezone`:** The tool message is everything the model will ever know about the weather. Open-Meteo's `current` block is bare numbers, and the same wind is `19.1` in km/h or `5.3` in m/s. The course code returned only that block, so the model guessed, and reported km/h as m/s and GMT as local time.

Full breakdown: [Exhaustive_3-tools.ipynb](Exhaustive_3-tools.ipynb).